## ASHRAE 140-2023

Section 9: CASE CE140 - Reduced outdoor dry bulb temperature at Low part-load ratio

In [1]:
import os

# Archived notebook: case files and weather paths are relative to the test folder
if os.path.basename(os.getcwd()) == "old_notebooks":
    os.chdir("..")
import opensimula as osm
import pandas as pd
import numpy as np

sim = osm.Simulation()
pro = sim.new_project("pro")
pro.read_json("cases/case_CE140.json")

Reading project data from dictonary
Reading completed.
Checking project: Case CE140
Checking completed.
Calculating solar direct shadows ...
Simulating Case CE140: ...


100%|██████████| 8760/8760 [00:18<00:00, 473.41step/s, n_iter=3]


In [2]:
system = pro.component("system")
sys_df=system.variable_dataframe()
space_df = pro.component("spaces_1").variable_dataframe()
sys_df["time_frac"] = sys_df["F_load"]/sys_df["efficiency_degradation"]
sys_df["power_vi"] = sys_df["indoor_fan_power"]
sys_df["power_ve"] = sys_df["time_frac"]*108
sys_df["power_comp"] = sys_df["power"]-sys_df["power_vi"]-sys_df["power_ve"]
sys_df["Sensible_space_load"] = space_df["system_sensible_heat"]
sys_df["Latent_space_load"] = space_df["system_latent_heat"]
sys_df["Total_space_load"] = sys_df["Latent_space_load"]+sys_df["Sensible_space_load"]
sys_df["abs_humidity"] = space_df["abs_humidity"]
sys_df["COP"] = -sys_df["Total_space_load"]/sys_df["power"]

                                                                 

mes_df_sum = sys_df.resample("ME", on='date').sum()
mes_df_mean = sys_df.resample("ME", on='date').mean()
mes_df_max = sys_df.resample("ME", on='date').max()
mes_df_min = sys_df.resample("ME", on='date').min()

excel = pd.DataFrame()
excel["Total_consumption"] = [mes_df_sum["power"].iloc[1]/1000]
excel["Compressor_consumption"] = [mes_df_sum["power_comp"].iloc[1]/1000]
excel["Indoor_fan_consumption"] = [mes_df_sum["power_vi"].iloc[1]/1000]
excel["Outdoor_fan_consumption"] = [mes_df_sum["power_ve"].iloc[1]/1000]
excel["Total_coil_load"] = [mes_df_sum["Q_total"].iloc[1]/1000]
excel["Sensible_coil_load"] = [mes_df_sum["Q_sensible"].iloc[1]/1000]
excel["Latent_coil_load"] = [mes_df_sum["Q_latent"].iloc[1]/1000]
excel["Total_space_load"] = [-mes_df_sum["Total_space_load"].iloc[1]/1000]
excel["Sensible_space_load"] = [-mes_df_sum["Sensible_space_load"].iloc[1]/1000]
excel["Latent_space_load"] = [-mes_df_sum["Latent_space_load"].iloc[1]/1000]


excel["COP"] = [mes_df_mean["COP"].iloc[1]]
excel["T_idb"] = [mes_df_mean["T_MA"].iloc[1]]
excel["w_i"] = [mes_df_mean["abs_humidity"].iloc[1]/1000]


excel["COP_max"] = [mes_df_max["COP"].iloc[1]]
excel["T_idb_max"] = [mes_df_max["T_MA"].iloc[1]]
excel["w_i_max"] = [mes_df_max["abs_humidity"].iloc[1]/1000]

excel["COP_min"] = [mes_df_min["COP"].iloc[1]]
excel["T_idb_min"] = [mes_df_min["T_MA"].iloc[1]]
excel["w_i_min"] = [mes_df_min["abs_humidity"].iloc[1]/1000]

excel.to_clipboard()
excel

,Total_consumption,Compressor_consumption,Indoor_fan_consumption,Outdoor_fan_consumption,Total_coil_load,Sensible_coil_load,Latent_coil_load,Total_space_load,Sensible_space_load,Latent_space_load,COP,T_idb,w_i,COP_max,T_idb_max,w_i_max,COP_min,T_idb_min,w_i_min
0,67.90649,55.96459,8.126144,3.815755,194.489168,194.489168,0.0,188.147557,188.147557,-0.0,2.770686,22.2,0.006225,2.770919,22.2,0.006225,2.770532,22.2,0.006225
